In [1]:
import findspark
findspark.init()

In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark=SparkSession.builder.getOrCreate()
# spark=SparkSession.builder.appName("learn").master("local[*]").getOrCreate()
spark

24/11/11 01:31:24 WARN Utils: Your hostname, Nishants-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
24/11/11 01:31:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/11 01:31:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
type(spark)

pyspark.sql.session.SparkSession

In [5]:
# help(spark.createDataFrame)

In [6]:
data=[(1,"Nishant"),(2,"ketu")]
df1=spark.createDataFrame(data=data,schema=['id','name'])
df1.show()

+---+-------+
| id|   name|
+---+-------+
|  1|Nishant|
|  2|   ketu|
+---+-------+



In [7]:
df1.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [8]:
from pyspark.sql.types import *
schema=StructType([StructField(name="id",dataType=IntegerType()),
            StructField(name="name",dataType=StringType())])

In [9]:
type(schema)

pyspark.sql.types.StructType

In [10]:
df2=spark.createDataFrame(data=data,schema=schema)
df2.show()

+---+-------+
| id|   name|
+---+-------+
|  1|Nishant|
|  2|   ketu|
+---+-------+



In [11]:
df2.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)



In [12]:
data=[{"id":1,"name":"nishant"},
      {"id":2,"name":"ketu"}
     ]
df3=spark.createDataFrame(data=data)
df3.show()

+---+-------+
| id|   name|
+---+-------+
|  1|nishant|
|  2|   ketu|
+---+-------+



In [13]:
#Reading CSV

In [14]:
df1=spark.read.csv(path='people-100.csv',header=True,inferSchema=True)
# df1=spark.read.csv(path='people-100.csv',header=True)
# without inferSchema everything is string
# use array of path for adding multiple files
# df=spark.read.csv(['people-100.csv','people-100-2.csv'],header=True,inferSchema=True)
# df=spark.read.csv('./csvfile/',header=True,inferSchema=True)
display(df1)
df1.show()

DataFrame[Index: int, User Id: string, First Name: string, Last Name: string, Sex: string, Email: string, Phone: string, Date of birth: date, Job Title: string]

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+--------------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|           Job Title|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+--------------------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|     Games developer|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24|      Phytotherapist|
|    3|DbeAb8CcdfeFC2c|  Kristine|   Travis|  Male|bthompson@example...|        277.609.7938|   1992-07-02|           Homeopath|
|    4|A31Bee3c201ef58|   Yesenia| Martinez|  Male|kaitlinkaiser@exa...|        584.094.6111|   2017-08-03|   Market researcher|
|    5|1bA7A3dc874da3c|      Lori|     Todd|  Male|buchananmanuel@ex...|   689-207-3558x7233|   1

In [15]:
# get number of partitions pyspark is creating
df1.rdd.getNumPartitions()
df1.printSchema()

root
 |-- Index: integer (nullable = true)
 |-- User Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Date of birth: date (nullable = true)
 |-- Job Title: string (nullable = true)



In [16]:
from pyspark.sql.types import *

schema= StructType().add(field="Index",data_type=IntegerType())\
                    .add(field="User Id",data_type=StringType())\
                    .add(field="First Name",data_type=StringType())\
                    .add(field="Last Name",data_type=StringType())\
                    .add(field="Sex",data_type=StringType())\
                    .add(field="Email",data_type=StringType())\
                    .add(field="Phone",data_type=StringType())\
                    .add(field="Date of birth",data_type=DateType())\
                    .add(field="Job Title",data_type=StringType())


df2=spark.read.csv(path='people-100.csv',schema=schema,header=True)
display(df2)
df2.show(2)

DataFrame[Index: int, User Id: string, First Name: string, Last Name: string, Sex: string, Email: string, Phone: string, Date of birth: date, Job Title: string]

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|      Job Title|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|Games developer|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24| Phytotherapist|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
only showing top 2 rows



In [17]:
#JSON files
# in json we use multiline, if multiline json is there, By default its False
from pyspark.sql.types import *

# Define the schema for the new DataFrame
schema2 = StructType().add("Id", LongType()).add("Name", StringType()).add("Age", IntegerType()).add("Email", StringType()).add("City", StringType()).add("Food", ArrayType(StringType()))
df_json=spark.read.json(path='people.json',multiLine=True)
display(df_json)
df_json.show()

#df_json=spark.read.json('./someFolder/*.json')
# This can help to take all json from a folder

DataFrame[age: bigint, city: string, email: string, food: array<string>, id: bigint, name: string]

+---+------------+-------------------+--------------------+---+-------+
|age|        city|              email|                food| id|   name|
+---+------------+-------------------+--------------------+---+-------+
| 30| Los Angeles|  alice@example.com|     [apple, banana]|  1|  Alice|
| 28|    New York|    bob@example.com|[orange, strawberry]|  2|    Bob|
| 32|     Chicago|charlie@example.com|  [grape, blueberry]|  3|Charlie|
| 29|     Houston|  david@example.com|   [pineapple, kiwi]|  4|  David|
| 27|     Phoenix|    eve@example.com|      [mango, peach]|  5|    Eve|
| 31|Philadelphia|  frank@example.com|  [pear, watermelon]|  6|  Frank|
| 26| San Antonio|  grace@example.com|     [apple, cherry]|  7|  Grace|
| 34|   San Diego|  henry@example.com|[banana, grapefruit]|  8|  Henry|
| 25|      Dallas| isabel@example.com| [pineapple, papaya]|  9| Isabel|
| 33|    San Jose|   jack@example.com|      [orange, plum]| 10|   Jack|
+---+------------+-------------------+--------------------+---+-

In [18]:
# pyspark saves df data in chunks, not in a single file
# mode can we ignore , error , append , overwrite
df_json.write.json("save_json.json",mode='overwrite')

In [19]:
df_new_json=spark.read.json("save_json.json")
df_new_json.show()

+---+------------+-------------------+--------------------+---+-------+
|age|        city|              email|                food| id|   name|
+---+------------+-------------------+--------------------+---+-------+
| 30| Los Angeles|  alice@example.com|     [apple, banana]|  1|  Alice|
| 28|    New York|    bob@example.com|[orange, strawberry]|  2|    Bob|
| 32|     Chicago|charlie@example.com|  [grape, blueberry]|  3|Charlie|
| 29|     Houston|  david@example.com|   [pineapple, kiwi]|  4|  David|
| 27|     Phoenix|    eve@example.com|      [mango, peach]|  5|    Eve|
| 31|Philadelphia|  frank@example.com|  [pear, watermelon]|  6|  Frank|
| 26| San Antonio|  grace@example.com|     [apple, cherry]|  7|  Grace|
| 34|   San Diego|  henry@example.com|[banana, grapefruit]|  8|  Henry|
| 25|      Dallas| isabel@example.com| [pineapple, papaya]|  9| Isabel|
| 33|    San Jose|   jack@example.com|      [orange, plum]| 10|   Jack|
+---+------------+-------------------+--------------------+---+-

In [20]:
csv_df=spark.read.csv(path='people-100.csv',header=True)

In [21]:
# show function
# default 20 rows and letter in columns
# truncate is True by default, not show full columns like email here
# vertical will change table to vertical form.
# to limit data we have limit, removes data

csv_df.show(3, truncate=False)
# csv_df.show(5, truncate=5,vertical=True)
csv_df.limit(2).show()

+-----+---------------+----------+---------+------+---------------------+----------------------+-------------+---------------+
|Index|User Id        |First Name|Last Name|Sex   |Email                |Phone                 |Date of birth|Job Title      |
+-----+---------------+----------+---------+------+---------------------+----------------------+-------------+---------------+
|1    |88F7B33d2bcf9f5|Shelby    |Terrell  |Male  |elijah57@example.net |001-084-906-7849x73518|1945-10-26   |Games developer|
|2    |f90cD3E76f1A9b9|Phillip   |Summers  |Female|bethany14@example.com|214.112.6044x4913     |1910-03-24   |Phytotherapist |
|3    |DbeAb8CcdfeFC2c|Kristine  |Travis   |Male  |bthompson@example.com|277.609.7938          |1992-07-02   |Homeopath      |
+-----+---------------+----------+---------+------+---------------------+----------------------+-------------+---------------+
only showing top 3 rows

+-----+---------------+----------+---------+------+--------------------+--------------

In [22]:
csv_df.printSchema()

root
 |-- Index: string (nullable = true)
 |-- User Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Date of birth: string (nullable = true)
 |-- Job Title: string (nullable = true)



In [23]:
# To call a column
# we can use 4 ways
# col('column')
# df.column
# df['column']
# expr('column')

# to select column
from pyspark.sql.functions import col,expr

csv_df.select(col("Index"),csv_df.Email,csv_df["Phone"],expr('sex')).show(2)


+-----+--------------------+--------------------+------+
|Index|               Email|               Phone|   sex|
+-----+--------------------+--------------------+------+
|    1|elijah57@example.net|001-084-906-7849x...|  Male|
|    2|bethany14@example...|   214.112.6044x4913|Female|
+-----+--------------------+--------------------+------+
only showing top 2 rows



In [24]:
# withColumn , drop , adding multiple column withColumns

from pyspark.sql.functions import col,lit

# casting,updating the colums
df1=csv_df.withColumn(colName='Index',col=col('Index').cast('Integer'))
df1.show(2)
# df1.printSchema()
df2=df1.withColumn(colName='Index',col=col("index")*5)
df2.show(2)

# creating new column, just give new name of column
df3=df2.withColumn(colName="country",col=lit("India"))
df3.show(2)
# df3.printSchema()

# to remove a colums
df3.drop("country").show(2)

# to add multiple columns
col_dict={"tax":lit(0),
          "fruit":lit("apple")}

df3.withColumns(col_dict)

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|      Job Title|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|Games developer|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24| Phytotherapist|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
only showing top 2 rows

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|

DataFrame[Index: int, User Id: string, First Name: string, Last Name: string, Sex: string, Email: string, Phone: string, Date of birth: string, Job Title: string, country: string, tax: int, fruit: string]

In [25]:
# withcolumRenamed
csv_df.withColumnRenamed('Index',"my_index").show(2)


+--------+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|my_index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|      Job Title|
+--------+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|       1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|Games developer|
|       2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24| Phytotherapist|
+--------+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
only showing top 2 rows



In [26]:
# struct type and struct field
# structtype structure StructType([StructField(name,dataType,nullabe)]), nullabe can be true or false

from pyspark.sql.types import *
data=[(1,("Nishant","ketu")),(2,("ajay","anand"))]
structName=StructType([StructField(name="first_name",dataType=StringType()),StructField(name="last_name",dataType=StringType())])
schema=StructType([StructField(name='id',dataType=IntegerType()),StructField(name='name',dataType=structName)])
df1=spark.createDataFrame(data,schema=schema)
# print beautifully
display(df1)
df1.show()

DataFrame[id: int, name: struct<first_name:string,last_name:string>]

+---+---------------+
| id|           name|
+---+---------------+
|  1|{Nishant, ketu}|
|  2|  {ajay, anand}|
+---+---------------+



In [27]:
 # string to schema , also we can do json to schema

schema_str = "name string , age int "
from pyspark.sql.types import _parse_datatype_string
spark_schema=_parse_datatype_string(schema_str)
spark_schema

StructType([StructField('name', StringType(), True), StructField('age', IntegerType(), True)])

In [28]:
#Array type
from pyspark.sql.types import *
data=[("nishant",[1,2,3,4,5]),("ketu",[1,4,3,4])]
schema=StructType([StructField("name",StringType()),StructField("my-num",ArrayType(IntegerType()))])
df1=spark.createDataFrame(data,schema)
df1=df1.withColumn("firstnum",col('my-num')[0])
df1.show()
df1.printSchema()

+-------+---------------+--------+
|   name|         my-num|firstnum|
+-------+---------------+--------+
|nishant|[1, 2, 3, 4, 5]|       1|
|   ketu|   [1, 4, 3, 4]|       1|
+-------+---------------+--------+

root
 |-- name: string (nullable = true)
 |-- my-num: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- firstnum: integer (nullable = true)



In [29]:
from pyspark.sql.functions import array
df2=df1.withColumn("merged_column",array(df1.firstnum,col('name')))
df2.show()

+-------+---------------+--------+-------------+
|   name|         my-num|firstnum|merged_column|
+-------+---------------+--------+-------------+
|nishant|[1, 2, 3, 4, 5]|       1| [1, nishant]|
|   ketu|   [1, 4, 3, 4]|       1|    [1, ketu]|
+-------+---------------+--------+-------------+



In [30]:
# explode ,it explore for every value of a array
# split  ,will split string into array
# Array_contain, will check an element in array

from pyspark.sql.functions import explode,col,split,array_contains

data=[("nishant",[6,7,3,4,5],'python,java'),("ketu",[8,9,3,4],'sql,dsa')]
df=spark.createDataFrame(data,['name','marks','skills'])
df.show()

# df.printSchema()
df1=df.withColumn('mark',explode(col('marks')))
df1.show()

df1=df.withColumn('skill',split(col('skills'),','))
df1.show()

df2=df1.withColumn('HaveDsa',array_contains(df1.skill,'dsa'))
df2.show()


+-------+---------------+-----------+
|   name|          marks|     skills|
+-------+---------------+-----------+
|nishant|[6, 7, 3, 4, 5]|python,java|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|
+-------+---------------+-----------+

+-------+---------------+-----------+----+
|   name|          marks|     skills|mark|
+-------+---------------+-----------+----+
|nishant|[6, 7, 3, 4, 5]|python,java|   6|
|nishant|[6, 7, 3, 4, 5]|python,java|   7|
|nishant|[6, 7, 3, 4, 5]|python,java|   3|
|nishant|[6, 7, 3, 4, 5]|python,java|   4|
|nishant|[6, 7, 3, 4, 5]|python,java|   5|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   8|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   9|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   3|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   4|
+-------+---------------+-----------+----+

+-------+---------------+-----------+--------------+
|   name|          marks|     skills|         skill|
+-------+---------------+-----------+--------------+
|nishant|[6, 7, 3, 4, 5]|python,java|[python, java]|

In [31]:
# mapType, like dict in python
# can create schema of mapType like MapType(StringType(),StringType())

data=[('ketu',{'color':'red','age':5}),('nishant',{'color':'blue','age':12})]
df=spark.createDataFrame(data,['name','specs'])
df.show(truncate=False)
df.printSchema()

df1=df.withColumn('age',col('specs')['age'])
df1.show(truncate=False)

+-------+--------------------------+
|name   |specs                     |
+-------+--------------------------+
|ketu   |{color -> red, age -> 5}  |
|nishant|{color -> blue, age -> 12}|
+-------+--------------------------+

root
 |-- name: string (nullable = true)
 |-- specs: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)

+-------+--------------------------+---+
|name   |specs                     |age|
+-------+--------------------------+---+
|ketu   |{color -> red, age -> 5}  |5  |
|nishant|{color -> blue, age -> 12}|12 |
+-------+--------------------------+---+



In [32]:
# explode for Map will make a key and val column
from pyspark.sql.functions import explode,map_keys,map_values

# Explode the 'specs' map into separate rows
df2 = df.select("name", explode(col("specs")).alias("spec_key", "spec_value"))
# Show result
# df2.show()

# mapkeys and map_values are similar
df.withColumn('k',map_values(df.specs)).show()


+-------+--------------------+----------+
|   name|               specs|         k|
+-------+--------------------+----------+
|   ketu|{color -> red, ag...|  [red, 5]|
|nishant|{color -> blue, a...|[blue, 12]|
+-------+--------------------+----------+



# Row class

In [33]:
# Row is a class and we can also make object
# Row object representing a single record in a DataFrame, similar to a row in a table. 
# Rows store data in fields that can be accessed by index or by column name.
# Creation: Automatically created by Spark when using createDataFrame().
# Field Access: Access data by name (row['column_name']) or index (row[index]).

from pyspark.sql import Row
# row = Row("nishant",1)
row = Row(name="nishant",Class=1)

row['name']
row.name
print(row)
row[0]


Row(name='nishant', Class=1)


'nishant'

In [34]:
# it is a row object, we can make df from this

row1=Row(name="pawan",Class=1)
row2=Row(name="mandal",Class=4)
row3=Row(name="juhi",Class=5)

df=spark.createDataFrame([row1,row2,row3])
df.show()

# we can also create a object form Row and use it
title=Row('name','roll')
row1=title('ramesh',22)
row2=title('suresh',87)
spark.createDataFrame([row1,row2]).show()

# we can also use row as nested data
data=[Row(name="mohit",prop=Row(color='black',age=77)),Row(name="sumit",prop=Row(color='purple',age=97))]
df=spark.createDataFrame(data)
df.show()
df.printSchema()


+------+-----+
|  name|Class|
+------+-----+
| pawan|    1|
|mandal|    4|
|  juhi|    5|
+------+-----+

+------+----+
|  name|roll|
+------+----+
|ramesh|  22|
|suresh|  87|
+------+----+

+-----+------------+
| name|        prop|
+-----+------------+
|mohit| {black, 77}|
|sumit|{purple, 97}|
+-----+------------+

root
 |-- name: string (nullable = true)
 |-- prop: struct (nullable = true)
 |    |-- color: string (nullable = true)
 |    |-- age: long (nullable = true)



# column class

In [35]:
from pyspark.sql.functions import lit,col

column=lit("apple")
print(type(column))
column

<class 'pyspark.sql.column.Column'>


Column<'apple'>

In [36]:
data=[("nishant",353,'python'),("ketu",958,'sql')]
df=spark.createDataFrame(data,['name','salary','skill'])

# accessing column
df.select(col('name'),lit('salary')).show()

+-------+------+
|   name|salary|
+-------+------+
|nishant|salary|
|   ketu|salary|
+-------+------+



# when and otherwise

In [37]:
data=[("nishant",353,'python','M'),("ketu",958,'sql','F'),("suraj",853,'berojgar','')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])
df.show()

from pyspark.sql.functions import when

df.select(df.name,df.salary,df.skill,when(df.gender=='M',"Male").when(df.gender=='F','Female')
          .otherwise("dont Know").alias("GENDER") ).show()

# if you not specify some value like then it will null, like if remove otherwise

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|      |
+-------+------+--------+------+

+-------+------+--------+---------+
|   name|salary|   skill|   GENDER|
+-------+------+--------+---------+
|nishant|   353|  python|     Male|
|   ketu|   958|     sql|   Female|
|  suraj|   853|berojgar|dont Know|
+-------+------+--------+---------+



# Alias, asc, desc, cast, like

In [38]:
df.select(df.name,col('salary').alias("aukat")).show() # eg alias

df.sort(df.name.desc()).show() # desc or asc

+-------+-----+
|   name|aukat|
+-------+-----+
|nishant|  353|
|   ketu|  958|
|  suraj|  853|
+-------+-----+

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|  suraj|   853|berojgar|      |
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
+-------+------+--------+------+



In [39]:
df.printSchema()
df.select(df.name,df.salary.cast('int')).printSchema()  # cast

root
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- skill: string (nullable = true)
 |-- gender: string (nullable = true)

root
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



In [40]:
df.show()
df.select(df.name.like('s%')).show()  # like 
df.filter(df.name.like('s%')).show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|      |
+-------+------+--------+------+

+------------+
|name LIKE s%|
+------------+
|       false|
|       false|
|        true|
+------------+

+-----+------+--------+------+
| name|salary|   skill|gender|
+-----+------+--------+------+
|suraj|   853|berojgar|      |
+-----+------+--------+------+



# Filter and where

In [41]:
df.filter(col('gender')=='M').show()

# we can write in String SQL expression
df.where("gender=='M'").show() 

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|nishant|   353|python|     M|
+-------+------+------+------+

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|nishant|   353|python|     M|
+-------+------+------+------+



# Distinct, Drop duplicate
distinct(): Removes duplicate rows based on all columns in the DataFrame.

df.distinct()

dropDuplicates(): Removes duplicate rows based on specific columns (if provided). otherwise work same

df.dropDuplicates(["column1", "column2"])

In [42]:
data=[("nishant",353,'python','M'),("ketu",958,'sql','F'),("suraj",853,'berojgar','M'),("ketu",958,'sql','F')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])
df.show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|   ketu|   958|     sql|     F|
+-------+------+--------+------+



In [43]:
df.distinct().show()

df.dropDuplicates(["gender"]).show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
+-------+------+--------+------+

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|   ketu|   958|   sql|     F|
|nishant|   353|python|     M|
+-------+------+------+------+



# orderBy and sort
work same

In [44]:
df.sort(df.gender,df.salary.desc()).show()

df.orderBy(df.gender,df.salary.desc()).show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|   ketu|   958|     sql|     F|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|nishant|   353|  python|     M|
+-------+------+--------+------+

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|   ketu|   958|     sql|     F|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|nishant|   353|  python|     M|
+-------+------+--------+------+



# union and union all
here both will do same thing, in pyspark it doesnot remove duplicate by union

work on same schema df

In [45]:
data1=[("nishant",353,'python','M'),("ketu",958,'sql','F')]
df1=spark.createDataFrame(data1,['name','salary','skill','gender'])

data2=[("suraj",853,'berojgar','M'),("ketu",958,'sql','F')]
df2=spark.createDataFrame(data2,['name','salary','skill','gender'])

df=df1.union(df2)
df.show()


+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|   ketu|   958|     sql|     F|
+-------+------+--------+------+



# groupBy, agg

In [46]:
# df.groupBy('gender').count().show()
df.groupBy('gender').max('salary').show()

from pyspark.sql.functions import max,count,max,min,sum
# df.groupBy('gender').agg(max('salary').alias('max_salary')).show()

df.groupBy('gender','skill').count().show()
df.groupBy('skill').agg(count('skill').alias('freq'),max('salary')).show()


+------+-----------+
|gender|max(salary)|
+------+-----------+
|     M|        853|
|     F|        958|
+------+-----------+



24/11/11 01:31:38 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+------+--------+-----+
|gender|   skill|count|
+------+--------+-----+
|     M|  python|    1|
|     F|     sql|    2|
|     M|berojgar|    1|
+------+--------+-----+

+--------+----+-----------+
|   skill|freq|max(salary)|
+--------+----+-----------+
|  python|   1|        353|
|     sql|   2|        958|
|berojgar|   1|        853|
+--------+----+-----------+



# unionByName
* union: Combines DataFrames, matching columns by position.
* unionByName: Combines DataFrames, matching columns by name.
* it also work when schema is different, but you need to mention allowMissingColumns

In [47]:
data1=[("nishant",353,'python','M'),("ketu",958,'sql','F')]
df1=spark.createDataFrame(data1,['name','salary','skill','gender'])

data2=[("suraj",7,'berojgar'),("ketu",9,'sql')]
df2=spark.createDataFrame(data2,['name','age','skill'])

df=df1.unionByName(df2,allowMissingColumns=True)
df.show()

+-------+------+--------+------+----+
|   name|salary|   skill|gender| age|
+-------+------+--------+------+----+
|nishant|   353|  python|     M|NULL|
|   ketu|   958|     sql|     F|NULL|
|  suraj|  NULL|berojgar|  NULL|   7|
|   ketu|  NULL|     sql|  NULL|   9|
+-------+------+--------+------+----+



# select

In [48]:
# df.select('name','salary').show
# df.select('*').show()
df.select([col for col in df.columns]).show()

+-------+------+--------+------+----+
|   name|salary|   skill|gender| age|
+-------+------+--------+------+----+
|nishant|   353|  python|     M|NULL|
|   ketu|   958|     sql|     F|NULL|
|  suraj|  NULL|berojgar|  NULL|   7|
|   ketu|  NULL|     sql|  NULL|   9|
+-------+------+--------+------+----+



# join
* same as SQL join (left,right,full,inner,leftsemi,leftanti, self)
* left semi is same inner but only column present in left table will come
* left anti is opps of left semi, it get non-matching row from left df

In [49]:
data1=[("nishant",353,'python','2'),("ketu",958,'sql','3'),("ram",648,'sql','9')]
df1=spark.createDataFrame(data1,['name','salary','skill','dept_id'])

data2=[(1,'IT'),(2,"sales"),(3,"HR")]
df2=spark.createDataFrame(data2,['dept_id','dept_name'])

df1.join(df2).show()

df1.join(df2, df1.dept_id==df2.dept_id , 'inner').show()

df1.join(df2, df1.dept_id==df2.dept_id , 'left').show()

df1.join(df2, df1.dept_id==df2.dept_id , 'leftsemi').show()

df1.join(df2, df1.dept_id==df2.dept_id , 'leftanti').show()


+-------+------+------+-------+-------+---------+
|   name|salary| skill|dept_id|dept_id|dept_name|
+-------+------+------+-------+-------+---------+
|nishant|   353|python|      2|      1|       IT|
|nishant|   353|python|      2|      2|    sales|
|nishant|   353|python|      2|      3|       HR|
|   ketu|   958|   sql|      3|      1|       IT|
|   ketu|   958|   sql|      3|      2|    sales|
|   ketu|   958|   sql|      3|      3|       HR|
|    ram|   648|   sql|      9|      1|       IT|
|    ram|   648|   sql|      9|      2|    sales|
|    ram|   648|   sql|      9|      3|       HR|
+-------+------+------+-------+-------+---------+

+-------+------+------+-------+-------+---------+
|   name|salary| skill|dept_id|dept_id|dept_name|
+-------+------+------+-------+-------+---------+
|nishant|   353|python|      2|      2|    sales|
|   ketu|   958|   sql|      3|      3|       HR|
+-------+------+------+-------+-------+---------+

+-------+------+------+-------+-------+---------

# pivot

In [59]:
data=[("nishant",353,'python','M'),("ketu",358,'sql','F'),("meh",153,'python','M'),("le",353,'sql','M'),("be",458,'sql','F')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])
df.show()
df.groupBy('skill','gender').count().show()
df.groupBy('skill').pivot('gender').count().show()

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|nishant|   353|python|     M|
|   ketu|   358|   sql|     F|
|    meh|   153|python|     M|
|     le|   353|   sql|     M|
|     be|   458|   sql|     F|
+-------+------+------+------+

+------+------+-----+
| skill|gender|count|
+------+------+-----+
|python|     M|    2|
|   sql|     F|    2|
|   sql|     M|    1|
+------+------+-----+

+------+----+---+
| skill|   F|  M|
+------+----+---+
|   sql|   2|  1|
|python|NULL|  2|
+------+----+---+

